In [4]:
# 1. MOUNT GOOGLE DRIVE & EXTRACT DATA LOCALLY
from google.colab import drive
import os

# Mount Drive (this will ask for permission)
drive.mount('/content/drive')

# --- UPDATE THESE PATHS ---
ZIP_PATH = '/content/drive/MyDrive/Proyecto_ML/x_train.zip'
CSV_PATH = '/content/drive/MyDrive/Proyecto_ML/y_train_v2.csv'
# --------------------------

# Local path where images are dumped
LOCAL_DATA_DIR = '/content/data_local'
IMG_FOLDER = LOCAL_DATA_DIR  # <--- CORRECCIÓN 1: Ruta directa

# Unzip data locally if it hasn't been unzipped yet
# Verificamos si existe alguna imagen para no volver a extraer
if not os.path.exists(os.path.join(LOCAL_DATA_DIR, 'img_1.png')):
    print("Extracting images locally for faster processing...")
    os.system(f"unzip -q {ZIP_PATH} -d {LOCAL_DATA_DIR}")
    print("Extraction complete!")
else:
    print("Images already extracted locally.")

# 2. DATA PREPROCESSING AND BASELINE TRAINING
import pandas as pd
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, classification_report

def load_data(csv_path, img_folder, img_size=(64, 64)):
    labels_df = pd.read_csv(csv_path)
    images = []
    labels = []

    for index, row in labels_df.iterrows():
        img_id = row['id']
        label = row['target']

        # <--- CORRECCIÓN 2: Añadido el prefijo "img_"
        img_path = os.path.join(img_folder, f"img_{img_id}.png")

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is not None:
            img = cv2.resize(img, img_size)
            img = img / 255.0
            images.append(img)
            labels.append(label)
        else:
            print(f"Warning: Could not read image {img_path}")

    return np.array(images), np.array(labels)

print("\nLoading and preprocessing data...")
X, y = load_data(CSV_PATH, IMG_FOLDER)

# Flatten images for classical ML (SVM requires 1D features)
X_flattened = X.reshape(X.shape[0], -1)

# Split 80/20 with stratification
X_train, X_val, y_train, y_val = train_test_split(
    X_flattened, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")

print("\nApplying PCA for dimensionality reduction...")
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

print("\nTraining SVM Baseline model (RBF Kernel)...")
svm_model = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(X_train_pca, y_train)

# 3. EVALUATION
print("\nEvaluating the model...")
y_pred = svm_model.predict(X_val_pca)

f1 = f1_score(y_val, y_pred, average='macro')
conf_matrix = confusion_matrix(y_val, y_pred)

print(f"\n--- RESULTS ---")
print(f"Validation F1-Score (Macro): {f1:.4f}")
print("Confusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", classification_report(y_val, y_pred))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Images already extracted locally.

Loading and preprocessing data...
Training set shape: (7380, 4096)
Validation set shape: (1846, 4096)

Applying PCA for dimensionality reduction...

Training SVM Baseline model (RBF Kernel)...

Evaluating the model...

--- RESULTS ---
Validation F1-Score (Macro): 0.2389
Confusion Matrix:
 [[ 81  88  70  53]
 [182 227 181 168]
 [155 178 136 132]
 [ 46  72  38  39]]

Classification Report:
               precision    recall  f1-score   support

           0       0.17      0.28      0.21       292
           1       0.40      0.30      0.34       758
           2       0.32      0.23      0.27       601
           3       0.10      0.20      0.13       195

    accuracy                           0.26      1846
   macro avg       0.25      0.25      0.24      1846
weighted avg       0.31      0.26      0.28      1846




## PROBLEMA 1

In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, classification_report

print("1. Preparing data for CNN...")
# CNNs require 2D images with a channel dimension.
# Reshape from (samples, 64, 64) to (samples, 64, 64, 1) for grayscale
X_cnn = X.reshape(-1, 64, 64, 1)

# Split data (using the same random_state to compare fairly with SVM)
from sklearn.model_selection import train_test_split
X_train_cnn, X_val_cnn, y_train, y_val = train_test_split(
    X_cnn, y, test_size=0.2, random_state=42, stratify=y
)

print(f"CNN Training set shape: {X_train_cnn.shape}")

print("\n2. Building CNN Architecture...")
model = models.Sequential([
    # First Convolutional Block (Extracts low-level features like edges)
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 1)),
    layers.MaxPooling2D((2, 2)), # Reduces spatial dimensions

    # Second Convolutional Block (Extracts higher-level shapes)
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Third Convolutional Block
    layers.Conv2D(64, (3, 3), activation='relu'),

    # Flatten to feed into dense layers
    layers.Flatten(),

    # Fully Connected Layer
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Regularization to prevent overfitting

    # Output Layer (4 classes, softmax for probabilities)
    layers.Dense(4, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy', # Used when labels are integers (0, 1, 2, 3)
              metrics=['accuracy'])

model.summary()

print("\n3. Training the CNN...")
# EarlyStopping prevents the model from training if validation loss stops improving
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train_cnn, y_train,
    epochs=30, # Max epochs, but early stopping might halt it sooner
    batch_size=32,
    validation_data=(X_val_cnn, y_val),
    callbacks=[early_stop]
)

print("\n4. Evaluating the CNN...")
# Predict probabilities and get the class with highest probability
y_pred_probs = model.predict(X_val_cnn)
y_pred_cnn = np.argmax(y_pred_probs, axis=1)

f1_cnn = f1_score(y_val, y_pred_cnn, average='macro')
conf_matrix_cnn = confusion_matrix(y_val, y_pred_cnn)

print(f"\n--- CNN RESULTS ---")
print(f"Validation F1-Score (Macro): {f1_cnn:.4f}")
print("Confusion Matrix:\n", conf_matrix_cnn)
print("\nClassification Report:\n", classification_report(y_val, y_pred_cnn))

1. Preparing data for CNN...
CNN Training set shape: (7380, 64, 64, 1)

2. Building CNN Architecture...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 62, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     1,179,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,236,036 (4.72 MB)

 Trainable params: 1,236,036 (4.72 MB)

 Non-trainable params: 0 (0.00 B)


3. Training the CNN...
Epoch 1/30
 88/231 ━━━━━━━━━━━━━━━━━━━━ 31s 219ms/step - accuracy: 0.3816 - loss: 1.2995

KeyboardInterrupt: 

## SOLUCION (Priemer entrenamineto que subir a keaggle)

In [12]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

print("1. Calculating Class Weights for Imbalanced Data...")
# IMPROVEMENT 1: Class Weights
# The dataset is imbalanced (e.g., class 1 has many more samples than class 3).
# compute_class_weight assigns a higher mathematical penalty to errors made on minority classes.
# This prevents the network from collapsing and predicting only the majority class.
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
print(f"Class weights applied: {class_weight_dict}")

print("\n2. Building Improved CNN Architecture...")
model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), padding='same', input_shape=(64, 64, 1)),
    # IMPROVEMENT 2: Batch Normalization
    # Normalizes the activations of the previous layer at each batch.
    # This stabilizes the learning process, allows the network to converge faster,
    # and helps avoid getting stuck in local minima (network collapse).
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    # IMPROVEMENT 3: Deeper Network architecture
    # Added a 128-filter convolutional layer to capture finer, more complex delay-Doppler spatial patterns.
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    # Dense Layers
    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    # IMPROVEMENT 4: Dropout (0.5)
    # Randomly ignores 50% of the neurons during training to prevent overfitting.
    # Forces the network to learn robust global features instead of memorizing the training data.
    layers.Dropout(0.5),

    layers.Dense(4, activation='softmax')
])

print("\n3. Compiling the CNN with custom Learning Rate...")
# IMPROVEMENT 5: Custom Learning Rate
# A slightly lower learning rate (0.0005 instead of default 0.001) prevents the optimizer
# from taking too large steps initially, which is a common cause for the network collapse we saw earlier.
adam_opt = optimizers.Adam(learning_rate=0.0005)
model.compile(optimizer=adam_opt,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("\n4. Training the CNN...")
# IMPROVEMENT 6: ReduceLROnPlateau Callback
# If the validation loss stops improving for 3 epochs, this callback cuts the learning rate in half.
# This helps the model "fine-tune" its weights when it gets close to the optimal minimum.
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

history = model.fit(
    X_train_cnn, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_cnn, y_val),
    class_weight=class_weight_dict, # Applying the calculated class weights here
    callbacks=[early_stop, reduce_lr]
)

print("\n5. Evaluating the Improved CNN...")
y_pred_probs = model.predict(X_val_cnn)
y_pred_cnn = np.argmax(y_pred_probs, axis=1)

f1_cnn = f1_score(y_val, y_pred_cnn, average='macro')
conf_matrix_cnn = confusion_matrix(y_val, y_pred_cnn)

print(f"\n--- NEW CNN RESULTS ---")
print(f"Validation F1-Score (Macro): {f1_cnn:.4f}")
print("Confusion Matrix:\n", conf_matrix_cnn)
print("\nClassification Report:\n", classification_report(y_val, y_pred_cnn))

1. Calculating Class Weights for Imbalanced Data...
Class weights applied: {np.int64(0): np.float64(1.5782720273738238), np.int64(1): np.float64(0.6093130779392338), np.int64(2): np.float64(0.768429820907955), np.int64(3): np.float64(2.3593350383631715)}

2. Building Improved CNN Architecture...

3. Compiling the CNN with custom Learning Rate...

4. Training the CNN...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
231/231 ━━━━━━━━━━━━━━━━━━━━ 118s 493ms/step - accuracy: 0.2369 - loss: 1.6732 - val_accuracy: 0.1056 - val_loss: 1.4215 - learning_rate: 5.0000e-04
Epoch 2/50
231/231 ━━━━━━━━━━━━━━━━━━━━ 109s 472ms/step - accuracy: 0.2516 - loss: 1.5709 - val_accuracy: 0.1203 - val_loss: 1.4058 - learning_rate: 5.0000e-04
Epoch 3/50
 19/231 ━━━━━━━━━━━━━━━━━━━━ 1:28 420ms/step - accuracy: 0.2803 - loss: 1.5772

KeyboardInterrupt: 

## PRIMERA SUBIDA

In [9]:
import pandas as pd
import numpy as np
import cv2
import os

print("1. Preparing Test Data...")
# --- UPDATE THESE PATHS TO MATCH YOUR GOOGLE DRIVE ---
TEST_ZIP_PATH = '/content/drive/MyDrive/Proyecto_ML/x_test.zip'
SAMPLE_SUB_PATH = '/content/drive/MyDrive/Proyecto_ML/y_test_submission_example_v2.csv'
# ---------------------------------------------------

TEST_LOCAL_DIR = '/content/test_data_local'

# Extract test images locally to avoid I/O bottleneck
if not os.path.exists(TEST_LOCAL_DIR):
    print("Extracting test images locally...")
    os.system(f"unzip -q {TEST_ZIP_PATH} -d {TEST_LOCAL_DIR}")
    print("Extraction complete!")
else:
    print("Test images already extracted.")

print("\n2. Loading Test Images based on Kaggle's required IDs...")
# Read the sample submission to get the exact IDs and order expected by Kaggle
submission_df = pd.read_csv(SAMPLE_SUB_PATH)

X_test_images = []
valid_ids = []

# Loop through the exact IDs requested by Kaggle
for img_id in submission_df['id']:
    # Remember the prefix "img_" we discovered earlier
    img_path = os.path.join(TEST_LOCAL_DIR, f"img_{img_id}.png")
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is not None:
        # Resize to match our CNN's input layer (64x64) and normalize
        img = cv2.resize(img, (64, 64))
        img = img / 255.0
        X_test_images.append(img)
        valid_ids.append(img_id)
    else:
        print(f"Warning: Could not read image {img_path}")

# Reshape for CNN input: (samples, 64, 64, 1)
X_test_cnn = np.array(X_test_images).reshape(-1, 64, 64, 1)

print("\n3. Generating Predictions...")
# Get probabilities for each class
y_test_probs = model.predict(X_test_cnn)
# Select the class with the highest probability
y_test_preds = np.argmax(y_test_probs, axis=1)

print("\n4. Formatting and Saving Submission File...")
# Create a new DataFrame formatted strictly for Kaggle
final_submission = pd.DataFrame({
    'id': valid_ids,
    'target': y_test_preds
})

# Save to a CSV file without the pandas index column
csv_filename = 'cnn_baseline_submission.csv'
final_submission.to_csv(csv_filename, index=False)

print(f"\nSUCCESS! File '{csv_filename}' has been created.")
print("Look in the left sidebar of Colab (Folder icon), download the CSV, and upload it to Kaggle!")

1. Preparing Test Data...
Extracting test images locally...
Extraction complete!

2. Loading Test Images based on Kaggle's required IDs...

3. Generating Predictions...


NameError: name 'model' is not defined